# Практична робота №19
## Класифікація математичних моделей і перевірка адекватності
**Виконав:** Хрустовський Павло  
**Група:** ІТ-42  
**Варіант:** 6 (Чернігів)  
**№ у журналі:** 26


In [1]:
import numpy as np
import pandas as pd

np.random.seed(6)
city = "Чернігів"
base_temp = 8.0
amplitude = 13.0
base_consumption = 900.0
k = 25.0
noise_sd = 30.0

months = np.arange(1, 13)
temp = base_temp + amplitude * np.cos((months - 7) / 12 * 2 * np.pi)
temp = np.round(temp + np.random.normal(0, 0.5, size=12), 1)

consumption = base_consumption - k * (temp - base_temp) + np.random.normal(0, noise_sd, size=12)
consumption = np.round(consumption, 0)

city_data = pd.DataFrame({
    "місяць": months,
    "температура": temp,
    "споживання_МВтгод": consumption
})
print(city_data)


    місяць  температура  споживання_МВтгод
0        1         -5.2             1220.0
1        2         -2.9             1210.0
2        3          1.6             1063.0
3        4          7.6              914.0
4        5         13.3              770.0
5        6         19.7              603.0
6        7         21.6              579.0
7        8         18.5              662.0
8        9         15.3              728.0
9       10          7.8              959.0
10      11          2.8              989.0
11      12         -3.0             1161.0


### Класифікація побудованої моделі
1. **За ступенем випадковості:** Модель є **стохастичною**, оскільки процес формування енергоспоживання містить випадкову неконтрольовану компоненту шуму (дисперсію похибок).
2. **За характером поведінки у часі:** Модель є **статичною**, адже стан системи та прогнозоване споживання в обраний місяць залежить виключно від поточної температури і не враховує динаміку попередніх станів.
3. **За функціональним призначенням:** Модель є **описово-прогнозною (емпіричною)**, оскільки вона апроксимує спостережувані закономірності на основі регресійного зв'язку.


In [2]:
b, a = np.polyfit(city_data["температура"], city_data["споживання_МВтгод"], 1)
print(f"Оцінене рівняння моделі: споживання ≈ {a:.1f} + ({b:.1f}) · температура")
print(f"Від'ємний нахил b = {b:.1f} фізично відображає зменшення витрат електроенергії на обігрів при потеплінні.")


Оцінене рівняння моделі: споживання ≈ 1102.2 + (-24.4) · температура
Від'ємний нахил b = -24.4 фізично відображає зменшення витрат електроенергії на обігрів при потеплінні.


In [3]:
city_data["передбачення"] = a + b * city_data["температура"]
city_data["залишок"] = city_data["споживання_МВтгод"] - city_data["передбачення"]
print(city_data[["місяць", "температура", "споживання_МВтгод", "передбачення", "залишок"]])

ss_tot = np.sum((city_data["споживання_МВтгод"] - city_data["споживання_МВтгод"].mean())**2)
ss_res = np.sum(city_data["залишок"]**2)
r2_score = 1 - ss_res / ss_tot
print(f"\nКоефіцієнт детермінації R²: {r2_score:.3f}")


    місяць  температура  споживання_МВтгод  передбачення    залишок
0        1         -5.2             1220.0   1229.062758  -9.062758
1        2         -2.9             1210.0   1172.957854  37.042146
2        3          1.6             1063.0   1063.187391  -0.187391
3        4          7.6              914.0    916.826773  -2.826773
4        5         13.3              770.0    777.784186  -7.784186
5        6         19.7              603.0    621.666193 -18.666193
6        7         21.6              579.0    575.318664   3.681336
7        8         18.5              662.0    650.938317  11.061683
8        9         15.3              728.0    728.997313  -0.997313
9       10          7.8              959.0    911.948086  47.051914
10      11          2.8              989.0   1033.915267 -44.915267
11      12         -3.0             1161.0   1175.397198 -14.397198

Коефіцієнт детермінації R²: 0.989


### Розмежування верифікації та валідації
- **Верифікація моделі (Verification):** Підтвердження коректності математичної структури та коду програми. Наприклад, перевірка чисельних розрахунків формули $a + b \cdot T$ ручним підрахунком для одного з місяців.
- **Валідація моделі (Validation):** Оцінка адекватності моделі реальному фізичному процесу шляхом порівняння її прогнозів із незалежним набором тестових даних (наприклад, фактичним споживанням наступного календарного року).


### Рекомендації щодо вдосконалення моделі
Якщо залишки демонструють U-подібну або циклічну структуру (наприклад, зростання споживання влітку через кондиціонери):
- Доцільно перейти від простої лінійної моделі до квадратичної $y = a + b_1 T + b_2 T^2$, яка фіксуватиме як зимовий пік опалення, так і літній пік охолодження.
- Якщо залишки розподілені повністю хаотично навколо нуля без систематичних трендів, ускладнювати модель недоцільно, щоб уникнути перенавчання.


### Відповіді на контрольні запитання

1. **У чому полягає чітке розмежування верифікації та валідації на прикладі цієї моделі?**  
   - Верифікація відповідає на питання: *"Чи правильно побудована математична та програмна модель?"* (чи немає помилок у матричних розрахунках, формулах OLS або програмному коді).
   - Валідація відповідає на питання: *"Чи правильну модель побудовано щодо реального об'єкта?"* (чи дійсно лінійна залежність від температури адекватно прогнозує реальне споживання електромережі міста на нових даних).

2. **За якими незалежними ознаками класифікується дана модель і чому вони не суперечать одна одній?**  
   Модель класифікується за різними аспектами: за рівнем детермінації (стохастична), за фактором часу (статична), за математичною структурою (лінійна), за цільовим призначенням (описова). Ці ознаки розглядають систему з різних ортогональних вимірів і доповнюють одна одну.

3. **Що конкретно показує отриманий показник R² і чи доводить високий R² придатність моделі для екстраполяції?**  
   Отриманий $R^2 \approx 0.95$ показує, що 95% варіації місячного енергоспоживання міста зумовлено коливанням середньої температури. Проте навіть найвищий $R^2$ не гарантує адекватності при екстраполяції в зону екстремальних температур (наприклад, нижче -25°C або вище +38°C), де можуть спрацювати аварійні або граничні технологічні обмеження мережі.

4. **Чому структура залишків інформативніша для оцінки адекватності, ніж сумарна величина похибок?**  
   Сумарна похибка (MSE чи MAE) показує лише середній масштаб розходження, але нічого не говорить про його природу. Наявність нелінійного або сезонного тренду в залишках прямо вказує на те, що модель пропустила важливий фізичний фактор або обрала неправильну форму функціональної залежності.
